In [17]:
import os
import random
import json

def analyze_directory(root_dir, sample_folder_count=1, sample_file_count=5):
    # 用于存储最终的结果
    result = {
        "root_directory": root_dir,
        "a_level_summary": {
            "total_a_folders": 0,
            "example_a_folders": [],
        },
        "sampled_a_folders": []
    }

    # 1. 计算根目录下的 A 级文件夹数量并打印
    a_level_folders = [f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))]
    result["a_level_summary"]["total_a_folders"] = len(a_level_folders)
    result["a_level_summary"]["example_a_folders"] = a_level_folders[:5]

    # 2. 随机采样 A 级文件夹
    sampled_a_folders = random.sample(a_level_folders, min(sample_folder_count, len(a_level_folders)))

    for sampled_a_folder in sampled_a_folders:
        sampled_a_path = os.path.join(root_dir, sampled_a_folder)
        sampled_a_info = {
            "a_folder_name": sampled_a_folder,
            "directory_tree": [],
            "sampled_files": []
        }

        # 3. 穷尽 A 级文件夹下的目录树
        file_list = []
        for root, dirs, files in os.walk(sampled_a_path):
            # 获取当前路径的相对路径和层级
            relative_root = os.path.relpath(root, sampled_a_path)
            folder_level = len(relative_root.split(os.sep)) if relative_root != "." else 0

            # 保存目录树信息
            sampled_a_info["directory_tree"].append({
                "level": folder_level,
                "folder_name": os.path.basename(root),
                "sub_folders": dirs,
                "file_count": len(files)
            })

            # 收集文件地址
            file_list.extend([os.path.join(root, f) for f in files])

        # 4. 随机采样末端文件
        sampled_files = random.sample(file_list, min(sample_file_count, len(file_list)))
        sampled_a_info["sampled_files"] = sampled_files

        # 添加到结果中
        result["sampled_a_folders"].append(sampled_a_info)

    # 将结果格式化为 JSON 并打印
    formatted_result = json.dumps(result, indent=4, ensure_ascii=False)
    print(formatted_result)

    # 可选择将结果保存到文件
    output_file = "directory_analysis.json"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(formatted_result)
    print(f"\n结果已保存到: {output_file}")

# 设置根目录
root_directory = "/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data"
analyze_directory(root_directory, sample_file_count=10, sample_folder_count=4)


{
    "root_directory": "/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data",
    "a_level_summary": {
        "total_a_folders": 3,
        "example_a_folders": [
            "BraTS2021_00000",
            "BraTS2021_00002",
            "BraTS2021_00003"
        ]
    },
    "sampled_a_folders": [
        {
            "a_folder_name": "BraTS2021_00002",
            "directory_tree": [
                {
                    "level": 0,
                    "folder_name": "BraTS2021_00002",
                    "sub_folders": [
                        "BraTS2021_00002_flair",
                        "BraTS2021_00002_seg",
                        "BraTS2021_00002_t1",
                        "BraTS2021_00002_t1ce",
                        "BraTS2021_00002_t2"
                    ],
                    "file_count": 0
                },
                {
                    "level": 1,
                    "folder_name": "BraTS2021_00002_flair",
                    "sub_folde

In [24]:
import requests
import csv
import re
import json
import os



def generate_metadata(root_directory, your_api_key=None):
    # DeepSeek API 的 URL 和 API 密钥
    DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"

    # 从环境变量中读取 API 密钥
    # 我推荐你到DeepSeek官网注册一个账号，然后在个人中心获取API_KEY，他们会给你你一辈子都用不完的额度
    # 获取之后填写到API = "你的API"中
    
    if your_api_key is None:
        if os.path.exists("config.py"):
            from config import API_KEY
            API_KEY = API_KEY

    # 读取 JSON 文件
    with open("directory_analysis.json", "r") as f:
        json_input = f.read()

    # 构建请求数据
    data = {
        "model": "deepseek-chat",
        "messages": [
            {
                "role": "system",
                "content": (
                    "你是一名熟练的数据科学家，善于解析复杂的文件目录并生成元数据表格。"
                    "你的任务是帮助用户分析医学影像数据集，并根据采样的文件结构生成metadata.csv。"
                    "你只需要输出带有恰当注释的python代码即可，多余的信息不输出。"
                )
            },
            {
                "role": "user",
                "content": (
                    f"我正在浏览一个医学影像数据集，它的根目录为：{json.loads(json_input)['root_directory']}。\n"
                    "这个数据集包含若干影像文件（可能包括多模态文件、单模态文件和掩码文件）。\n"
                    "我采样了一些子文件夹（记为 A 级文件夹）以及其中的 B/C 级文件夹，目录树和采样文件的信息如下：\n"
                    f"{json_input}\n"
                    "我需要你：\n"
                    "1. 分析文件命名的规律，判断是否存在多模态文件或掩码文件。\n"
                    "2. 根据这些规律生成构建 metadata.csv 的 Python 代码。\n"
                    "3. 输出的代码应该以根目录为输入，生成的 csv 应保存在根目录下，csv 的列包括 sample_id（若没有明显 id，则直接用数字序号）、各模态的文件地址（如 flair_path, t1_path 等，若不存在则为空，若没有明显的多模态特征那么记为image_path）、以及掩码地址（若不存在则为空）。"
                )
            }
        ],
        "stream": False
    }

    # 发送请求
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data)

    # 检查响应状态码
    if response.status_code == 200:
        result = response.json()
        try:
            # 打印模型输出的原始内容
            print("模型输出的原始内容：")
            model_output = result["choices"][0]["message"]["content"]
            print(model_output)

            # 保存 LLM 输出到 result.json
            with open("result.json", "w") as f:
                f.write(model_output)
            print("LLM 的响应内容已保存到 result.json 文件中。")

            # 尝试从 LLM 的输出中提取生成的代码
            code_match = re.search(r"```python(.*?)```", model_output, re.DOTALL)
            if code_match:
                extracted_code = code_match.group(1).strip()
                with open("generate_metadata.py", "w") as f:
                    f.write(extracted_code)
                print("生成的 Python 代码已保存到 generate_metadata.py 文件中。")
            else:
                print("未检测到有效的 Python 代码块，请手动检查 LLM 输出。")
        except Exception as e:
            print(f"解析响应内容时发生错误：{e}")
    else:
        print(f"请求失败，状态码：{response.status_code}")
        print(response.text)


generate_metadata(root_directory, your_api_key=None)

模型输出的原始内容：
```python
import os
import csv

def generate_metadata(root_directory):
    metadata = []
    sample_id = 0

    for a_folder in os.listdir(root_directory):
        a_folder_path = os.path.join(root_directory, a_folder)
        if os.path.isdir(a_folder_path):
            flair_path = t1_path = t1ce_path = t2_path = seg_path = ""

            for sub_folder in os.listdir(a_folder_path):
                sub_folder_path = os.path.join(a_folder_path, sub_folder)
                if os.path.isdir(sub_folder_path):
                    if sub_folder.endswith("_flair"):
                        flair_path = os.path.join(sub_folder_path, os.listdir(sub_folder_path)[0])
                    elif sub_folder.endswith("_t1"):
                        t1_path = os.path.join(sub_folder_path, os.listdir(sub_folder_path)[0])
                    elif sub_folder.endswith("_t1ce"):
                        t1ce_path = os.path.join(sub_folder_path, os.listdir(sub_folder_path)[0])
                    

In [25]:
import os
import csv
import subprocess

def execute_metadata_script(root_directory):
    metadata_file = os.path.join(root_directory, "metadata.csv")
    script_file = "generate_metadata.py"

    # 检查 generate_metadata.py 是否存在
    if not os.path.exists(script_file):
        print(f"脚本 {script_file} 不存在，请确保文件已正确生成。")
    else:
        # 执行 generate_metadata.py 脚本
        print(f"正在执行 {script_file}...")
        result = subprocess.run(["python", script_file], capture_output=True, text=True)

        # 检查执行结果
        if result.returncode == 0:
            print(f"{script_file} 执行成功！")
        else:
            print(f"{script_file} 执行失败！")
            print(f"错误输出：\n{result.stderr}")

        # 检查 metadata.csv 是否存在
        if os.path.exists(metadata_file):
            print(f"metadata.csv 文件已生成，路径为：{metadata_file}")

            # 打印 metadata.csv 的前 5 行
            try:
                with open(metadata_file, "r") as f:
                    reader = csv.reader(f)
                    print("metadata.csv 的前 5 行内容：")
                    for i, row in enumerate(reader):
                        print(row)
                        if i == 4:  # 打印前 5 行
                            break
            except Exception as e:
                print(f"读取 metadata.csv 时发生错误：{e}")
        else:
            print("metadata.csv 文件未生成，请检查脚本逻辑和根目录路径。")

execute_metadata_script(root_directory)

正在执行 generate_metadata.py...
generate_metadata.py 执行成功！
metadata.csv 文件已生成，路径为：/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/metadata.csv
metadata.csv 的前 5 行内容：
['sample_id', 'flair_path', 't1_path', 't1ce_path', 't2_path', 'seg_path']
['0', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_flair/00000057_brain_flair.nii', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t1/00000057_brain_t1.nii', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t1ce/00000057_brain_t1ce.nii', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_t2/00000057_brain_t2.nii', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00000/BraTS2021_00000_seg/00000057_final_seg.nii']
['1', '/teamspace/studios/this_studio/PreProcPipe/BraTS2021_Training_Data/BraTS2021_00002/